# Greek Medical Report Anonymization

This notebook launches the anonymization web app inside Google Colab.

Run the cells from top to bottom.

Before starting, make sure the exported model folder is available in Google Drive.


In [ ]:
%cd /content
!rm -rf greek-medical-anonymization
!git clone https://github.com/VanessaLislevand/greek-medical-anonymization.git
%cd /content/greek-medical-anonymization


In [ ]:
%pip install -e ".[ml,ui]"


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil

repo_root = Path('/content/greek-medical-anonymization')
repo_model_dir = repo_root / 'models' / 'xlmr_phi_final'

# Change this path if your model is stored elsewhere in Google Drive.
DRIVE_MODEL_DIR = '/content/drive/MyDrive/xlmr_phi_final'
drive_model_dir = Path(DRIVE_MODEL_DIR)

print(f'Repository root: {repo_root}')
print(f'Drive model path: {drive_model_dir}')
print(f'Model folder exists in Drive: {drive_model_dir.exists()}')

if not drive_model_dir.exists():
    raise FileNotFoundError(
        'Model folder not found in Google Drive. Update DRIVE_MODEL_DIR and run this cell again.'
    )

if repo_model_dir.exists() or repo_model_dir.is_symlink():
    if repo_model_dir.is_symlink() or repo_model_dir.is_file():
        repo_model_dir.unlink()
    else:
        shutil.rmtree(repo_model_dir)

repo_model_dir.symlink_to(drive_model_dir, target_is_directory=True)

print('Model files:')
for path in sorted(repo_model_dir.iterdir()):
    if path.is_file():
        print(f'- {path.name}')


In [ ]:
import os
import subprocess
import sys
import time

from google.colab import output

os.chdir('/content/greek-medical-anonymization')

try:
    streamlit_process
except NameError:
    streamlit_process = None

if streamlit_process is not None and streamlit_process.poll() is None:
    print('The web app is already running.')
else:
    streamlit_process = subprocess.Popen(
        [
            sys.executable,
            '-m',
            'streamlit',
            'run',
            'src/greek_med_anonymizer/web_app.py',
            '--server.port',
            '8501',
            '--server.headless',
            'true',
        ]
    )
    time.sleep(5)
    output.serve_kernel_port_as_window(8501)


In [ ]:
# Optional: stop the running web app.
if 'streamlit_process' in globals() and streamlit_process is not None and streamlit_process.poll() is None:
    streamlit_process.terminate()
    print('The web app was stopped.')
else:
    print('No running web app process was found.')
